# 02 - Converting a FHIR Message into a FHIR Document for the NHS England API

`01-oru-r01-to-fhir-message.ipynb` turned `ORU^R01` messages into FHIR Messages (`examples/fhir/message/`).
This notebook turns each FHIR Message into the **FHIR Document** that laboratory reporting systems POST to the
[NHS England Pathology and Laboratory Medicine Reporting FHIR API](https://digital.nhs.uk/developer/api-catalogue/pathology-laboratory-reporting)
(`POST /FHIR/R4/Bundle`), following
[FHIR Message to FHIR Document](https://kevinmayfield.github.io/PathologyAndLaboratory/mapping.html#fhir-message-to-fhir-document)
and the [NHS England Pathology Laboratory Reporting API](https://kevinmayfield.github.io/PathologyAndLaboratory/PathologyReportingAPI.html)
page.

A message is an *event* ("here is a result"); a document is a *record* - a complete, attested report that is
stored (in Patient Data Manager) and read as a whole. So the document adds:

1. a **Composition** - the document header and its sections
2. **human readable narrative** - the XHTML a person sees, alongside the computer processable resources
3. a **Provenance** naming the sending system
4. the NHS England rules: `type` `document`, Composition first, exactly one DiagnosticReport, the patient's NHS
   Number in `Composition.subject.identifier`, the requesting ODS code reachable from the ServiceRequest, and a
   permitted `meta.profile` on every resource

In [1]:
import html
import json
import os
import subprocess
import uuid
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IN_DIR = ROOT / "examples" / "fhir" / "message"
OUT_DIR = ROOT / "examples" / "fhir" / "document"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCT = "http://snomed.info/sct"
LOINC = "http://loinc.org"
NHS_NUMBER = "https://fhir.nhs.uk/Id/nhs-number"
ODS = "https://fhir.nhs.uk/Id/ods-organization-code"

# meta.profile values the NHS England API permits - versions as published in the
# fhir.r4.nhsengland.pathology 0.3.0 package (the API specification cites
# England-Observation-Lab-Pathology 0.3.0, which is not yet in a published package)
NHSE_PROFILE = {
    "Bundle": "https://fhir.nhs.uk/England/StructureDefinition/England-Bundle-Lab-Pathology|0.1.0",
    "Composition": "http://hl7.eu/fhir/laboratory/StructureDefinition/Composition-eu-lab|2.0.0",
    "DiagnosticReport": "http://hl7.eu/fhir/laboratory/StructureDefinition/DiagnosticReport-eu-lab|2.0.0-nhs-fix",
    "Observation": "https://fhir.nhs.uk/England/StructureDefinition/England-Observation-Lab-Pathology|0.2.1",
    "ServiceRequest": "http://hl7.eu/fhir/laboratory/StructureDefinition/ServiceRequest-eu-lab|2.0.0",
    "Specimen": "http://hl7.eu/fhir/laboratory/StructureDefinition/Specimen-eu-lab|2.0.0",
    "Patient": "http://hl7.eu/fhir/base/StructureDefinition/patient-eu-core|2.0.0",
    "Practitioner": "https://fhir.hl7.org.uk/StructureDefinition/UKCore-Practitioner|2.4.0",
    "PractitionerRole": "https://fhir.hl7.org.uk/StructureDefinition/UKCore-PractitionerRole|2.4.0",
    "Organization": "https://fhir.hl7.org.uk/StructureDefinition/UKCore-Organization|2.5.0",
    "Provenance": "https://fhir.nhs.uk/England/StructureDefinition/England-Provenance-Lab-Pathology|0.1.0",
}
ORGANIZER = "http://hl7.org/fhir/6.0/StructureDefinition/extension-Observation.organizer"
COMPOSITION_R5 = "http://hl7.org/fhir/5.0/StructureDefinition/extension-DiagnosticReport.composition"

# The sending product: Provenance.agent.who.identifier.value = <Product ID>:<(Optional) Test Reporter>
PRODUCT_ID = "EXAMPLE-PRODUCT"

messages = {p.stem: json.loads(p.read_text()) for p in sorted(IN_DIR.glob("*.json"))}
list(messages)

['CRP-Report-Example',
 'FBC-Report-Example',
 'GTT-Unstructured-Report-Example',
 'HBsAg-Report-Example',
 'HPV-Report-Example',
 'HbA1c-Report-Example',
 'LFT-UandE-Report-Example',
 'Lipids-HbA1c-Report-Example',
 'Urine-MCS-Unstructured-Report-Example']

## 1. Human readable and computer processable

A FHIR resource has two parts:

- the **computer processable** part - the elements (`code`, `valueQuantity`, `referenceRange`, ...) that a
  receiving system stores, graphs, decision-supports and files against the patient
- the **human readable** part - `text` (the *narrative*), a fragment of XHTML that shows a person what the
  resource says. In a Composition, each `section.text` is the narrative of that part of the report, and
  `Composition.text` is the narrative of the document header

For a FHIR Document the narrative is the clinically **attested** content: it is what a clinician sees when the
receiving system displays the document, and what is reproduced if the document is printed or forwarded. A
receiver that does not understand every coded element can still show the report safely. The two parts **SHALL**
say the same thing - a narrative that disagrees with the data (a value rounded differently, a missing flag or
comment) is a clinical safety issue.

The NHS England examples leave the section narrative `empty`. A laboratory reporting system creating a document
from an `ORU^R01` has all it needs to generate it, so this notebook does. Tips for generating the XHTML:

- **Generate it from the FHIR resources you have just built**, not from the HL7 v2 text, so the narrative
  cannot drift from the data. Set `text.status` = `generated`
- **Escape everything** taken from the message (`<`, `>`, `&`) - result values such as `<5` or `>35` and free
  text comments will otherwise break the XHTML or inject markup
- **Use the FHIR XHTML subset**: the root is `<div xmlns="http://www.w3.org/1999/xhtml">`; use simple elements
  (`p`, `table`, `tr`, `th`, `td`, `b`, `i`, `br`, `ul`, `li`, `span`); no `script`, `style` elements, forms,
  event attributes or external images
- **Lay out results as the laboratory reports them**: one table per test group (OBR), one row per result (OBX)
  with test, result, units, reference range and flag, then the result comments (NTE)
- **Keep line breaks and spacing** of text results (the unstructured GTT and urine MCS examples): HL7 v2
  `\.br\` became new lines in `valueString`; show them with `<br/>` or a `<pre>` block, otherwise columns
  of text results run together
- **Show abnormal flags in words** ("Low", "High") as well as with emphasis - do not rely on colour
- **Put the identifying details in `Composition.text`**: patient NHS Number, report number (OBR-3), requester,
  laboratory, specimen collected / received, report issued time and status (preliminary / final / corrected)
- **Keep it compact** - the narrative is a summary of the coded content, not a rendering of the whole message

In [2]:
FLAG = {"L": "Low", "H": "High", "LL": "Critically low", "HH": "Critically high", "A": "Abnormal",
        "AA": "Critically abnormal", "N": ""}

def e(text):
    return html.escape(str(text), quote=False)

def multiline(text):
    return "<br/>".join(e(line) for line in str(text).split("\n"))

def display(concept):
    if not concept:
        return ""
    return concept.get("text") or concept.get("coding", [{}])[0].get("display") or concept.get("coding", [{}])[0].get("code", "")

def number(v):
    return f"{v:g}" if isinstance(v, float) else str(v)

def value_text(obs):
    if "valueQuantity" in obs:
        q = obs["valueQuantity"]
        return f"{e(q.get('comparator', ''))}{number(q['value'])}", e(q.get("unit", ""))
    if "valueRange" in obs:
        r = obs["valueRange"]
        return f"{number(r['low']['value'])} - {number(r['high']['value'])}", e(r["low"].get("unit", ""))
    if "valueCodeableConcept" in obs:
        return e(display(obs["valueCodeableConcept"])), ""
    if "valueString" in obs:
        return multiline(obs["valueString"]), ""
    return "", ""

def range_text(obs):
    rr = (obs.get("referenceRange") or [{}])[0]
    if "low" in rr and "high" in rr:
        return f"{number(rr['low']['value'])} - {number(rr['high']['value'])}"
    if "high" in rr:
        return f"&lt; {number(rr['high']['value'])}"
    if "low" in rr:
        return f"&gt; {number(rr['low']['value'])}"
    return multiline(rr.get("text", ""))

def flag_text(obs):
    code = (obs.get("interpretation") or [{}])[0].get("coding", [{}])[0].get("code", "")
    return f"<b>{FLAG.get(code, code)}</b>" if FLAG.get(code, code) else ""

def results_table(observations):
    rows = ["<tr><th>Test</th><th>Result</th><th>Units</th><th>Reference range</th><th>Flag</th></tr>"]
    comments = []
    for obs in observations:
        value, units = value_text(obs)
        rows.append(f"<tr><td>{e(display(obs['code']))}</td><td>{value}</td><td>{units}</td>"
                    f"<td>{range_text(obs)}</td><td>{flag_text(obs)}</td></tr>")
        comments += [f"<p><i>{e(display(obs['code']))}</i>: {multiline(n['text'])}</p>" for n in obs.get("note", [])]
    return "<table>" + "".join(rows) + "</table>" + "".join(comments)

def narrative(inner):
    return {"status": "generated", "div": f'<div xmlns="http://www.w3.org/1999/xhtml">{inner}</div>'}

## 2. Looking up practitioners and organisations

The FHIR Messages mirror HL7 v2: practitioners and organisations are **identifier references** (ODS code, GMC
or GMP code, and a display name) - there are no Practitioner or Organization resources in them. The NHS England
document needs the resources (`ServiceRequest.requester` → PractitionerRole → Organization, and the laboratory as
author, custodian and performer), so they are looked up here and added to the document.

Where to look them up:

- **Organisations (ODS codes)** - the NHS England Organisation Data Service: the
  [ODS ORD API](https://digital.nhs.uk/services/organisation-data-service/organisation-data-service-apis)
  (`https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/{ODS code}`, public), the
  [ODS FHIR API (Organisation Data Terminology)](https://digital.nhs.uk/developer/api-catalogue/organisation-data-terminology),
  or the [ODS CSV downloads](https://digital.nhs.uk/services/organisation-data-service/data-search-and-export/csv-downloads)
  (for example `epraccur.csv` for GP practices and `etr.csv` for NHS trusts)
- **GP practitioners (GMP codes)** - the ODS CSV download `egpcur.csv`
- **GMC, NMC and HCPC numbers** - the professional registers; most NHS systems hold these in their own user or
  practitioner directory (**To confirm** a national API)

A production system would cache a directory (refreshed from the ODS downloads or API) rather than call an API for
every report. Here `DIRECTORY` holds the entries for these examples (taken from the NHS England examples), and
`ods_lookup` shows the live ODS ORD API being used to fill gaps.

In [3]:
import urllib.request

DIRECTORY = {
    ("Organization", ODS, "Y8J7D"): {"name": "TD008362 PATH LAB 001",
        "address": {"line": ["PATHOLOGY LAB", "7-8 WELLINGTON PLACE"], "city": "LEEDS",
                    "district": "WEST YORKSHIRE", "postalCode": "LS1 4AP"}},
    ("Organization", ODS, "B82033"): {"name": "PICKERING MEDICAL PRACTICE",
        "address": {"line": ["SOUTHGATE"], "city": "PICKERING", "district": "NORTH YORKSHIRE", "postalCode": "YO18 8BL"}},
    ("Practitioner", "https://fhir.hl7.org.uk/Id/gmc-number", "C1008215"): {
        "name": {"use": "official", "family": "GASKELL", "given": ["Gale"], "prefix": ["Dr"]}},
}

def ods_lookup(ods_code):
    """The ODS ORD API (public) - name and address of an organisation, or None."""
    try:
        url = f"https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/{ods_code}"
        with urllib.request.urlopen(url, timeout=20) as response:
            org = json.load(response)["Organisation"]
        loc = org["GeoLoc"]["Location"]
        return {"name": org["Name"],
                "address": {"line": [loc[k] for k in ("AddrLn1", "AddrLn2", "AddrLn3") if loc.get(k)],
                            "city": loc.get("Town"), "district": loc.get("County"), "postalCode": loc.get("PostCode")}}
    except Exception:
        return None

def resolve(reference):
    """Identifier reference -> (fullUrl, resource) from the directory, falling back to ODS for organisations."""
    kind, ident = reference["type"], reference["identifier"]
    entry = DIRECTORY.get((kind, ident["system"], ident["value"]))
    if entry is None and kind == "Organization":
        entry = ods_lookup(ident["value"])
    resource_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f"{ident['system']}|{ident['value']}"))
    resource = {"resourceType": kind, "id": resource_id, "identifier": [ident]}
    if kind == "Organization":
        resource["name"] = (entry or {}).get("name", reference.get("display", ident["value"]))
        if entry and entry.get("address"):
            resource["address"] = [{k: v for k, v in entry["address"].items() if v}]
    else:
        resource["name"] = [(entry or {}).get("name") or {"text": reference.get("display", ident["value"])}]
    return f"urn:uuid:{resource_id}", resource

# The live ODS directory for the codes in these examples. Y8J7D is a real ODS code: the NHS England examples
# gave it an illustrative laboratory name, so the directory entry above (from the examples) is used first.
{code: (ods_lookup(code) or {}).get("name") for code in ["Y8J7D", "B82033"]}

{'Y8J7D': 'NIGHTINGALE HOSPITAL BRISTOL',
 'B82033': 'PICKERING MEDICAL PRACTICE'}

## 3. Building the document

For each DiagnosticReport in the message (there is one per report - NHS England requires exactly one per
document):

1. resolve the identifier references (section 2), then collect the DiagnosticReport and every resource it
   references, directly or indirectly
2. create the **Composition**: `identifier` = the report number (OBR-3) and `type` sharing a coding with
   `DiagnosticReport.code` (both required by HL7 Europe `Bundle-eu-lab`), `subject.identifier` = the NHS Number,
   `author` / `custodian` / `attester` = the laboratory, one `section` per test group with generated narrative
3. link the DiagnosticReport to the Composition (R5 `DiagnosticReport.composition` extension, as
   `DiagnosticReport-eu-lab` requires) and mark test groups with the NHS England *organizer* extension
4. add a **Provenance** targeting the Composition (`iso-21089-lifecycle#transform`, as the NHS England examples)
5. set the NHS England `meta.profile` on every resource, put the Composition first, and drop the MessageHeader

In [4]:
SPECIALTY = {  # HL7 v2 table 0074 (OBR-24) -> HL7 Europe lab-specialty (SNOMED CT)
    "CH": ("394596001", "Chemical pathology"), "HM": ("394916005", "Hematopathology"),
    "MB": ("408454008", "Clinical microbiology"), "SR": ("394598000", "Immunopathology"),
    "IMM": ("394598000", "Immunopathology"), "BLB": ("421661004", "Blood banking and transfusion medicine (specialty)")}
GENERAL = ("394915009", "General pathology (specialty)")

def doc_id(*parts):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, "document|" + "|".join(parts)))

def references(resource):
    """Every reference in a resource (any depth)."""
    out = []
    def walk(x):
        if isinstance(x, dict):
            if isinstance(x.get("reference"), str):
                out.append(x["reference"])
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)
    walk(resource)
    return out

def resolve_references(by_url):
    """Replace identifier references to Practitioner / Organization with references to looked-up resources."""
    found = {}
    def walk(x):
        if isinstance(x, dict):
            if x.get("type") in ("Organization", "Practitioner") and "identifier" in x and "reference" not in x:
                url, resource = resolve(x)
                found[url] = resource
                x["reference"] = url
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for v in x:
                walk(v)
    walk(list(by_url.values()))
    by_url.update(found)

def to_document(message, name):
    message = json.loads(json.dumps(message))
    by_url = {en["fullUrl"]: en["resource"] for en in message["entry"]}
    resolve_references(by_url)
    header = message["entry"][0]["resource"]
    documents = []
    for focus in header["focus"]:
        report_url = focus["reference"]
        report = json.loads(json.dumps(by_url[report_url]))

        # 1. the report and everything it references
        included, queue = [report_url], [report_url]
        while queue:
            for r in references(by_url[queue.pop(0)]):
                if r in by_url and r not in included and by_url[r]["resourceType"] != "MessageHeader":
                    included.append(r)
                    queue.append(r)
        resources = {u: (report if u == report_url else json.loads(json.dumps(by_url[u]))) for u in included}

        lab_ref = report["performer"][0]
        lab = resources[lab_ref["reference"]]
        nhs_number = report["subject"]["identifier"]["value"]
        category = report["category"][0]["coding"][0]["code"]
        specialty = GENERAL  # as the NHS England examples; SPECIALTY[category] would be more specific

        # 2. Composition with one section per test group
        composition_id = doc_id(report["identifier"][0].get("system", ""), report["identifier"][0]["value"])
        composition_url = f"urn:uuid:{composition_id}"
        sections = []
        for result in report["result"]:
            top = resources[result["reference"]]
            members = [resources[m["reference"]] for m in top.get("hasMember", [])] or [top]
            sections.append({"title": display(top["code"]), "code": top["code"],
                             "text": narrative(results_table(members)), "entry": [result]})
        if report.get("conclusion"):
            sections.append({"title": "Report comment",
                             "code": {"coding": [{"system": LOINC, "code": "48767-8", "display": "Annotation comment [Interpretation] Narrative"}]},
                             "text": narrative(f"<p>{multiline(report['conclusion'])}</p>")})

        requests = [resources[b["reference"]] for b in report.get("basedOn", [])]
        role = resources.get(requests[0]["requester"]["reference"]) if requests else None
        gp = resources.get(role["practitioner"]["reference"]) if role else None
        gp_org = resources.get(role["organization"]["reference"]) if role else None
        specimens = [resources[s["reference"]] for s in report.get("specimen", [])]
        header_rows = [
            ("NHS Number", e(nhs_number)),
            ("Report number", e(report["identifier"][0]["value"])),
            ("Status", e(report["status"])),
            ("Requested by", e(" ".join(filter(None, [*(gp or {}).get("name", [{}])[0].get("prefix", []),
                                                   *(gp or {}).get("name", [{}])[0].get("given", []),
                                                   (gp or {}).get("name", [{}])[0].get("family", "")])))
             + (f", {e(gp_org['name'])} ({e(gp_org['identifier'][0]['value'])})" if gp_org else "")),
            ("Laboratory", f"{e(lab['name'])} ({e(lab['identifier'][0]['value'])})"),
            ("Specimen", "; ".join(f"{e(display(s.get('type')))} collected {e(s['collection'].get('collectedDateTime', ''))}"
                                   for s in specimens)),
            ("Issued", e(report.get("issued", ""))),
        ]
        composition = {
            "resourceType": "Composition", "id": composition_id,
            "meta": {"profile": [NHSE_PROFILE["Composition"]]},
            "text": narrative("<table>" + "".join(f"<tr><th>{k}</th><td>{v}</td></tr>" for k, v in header_rows) + "</table>"),
            "identifier": report["identifier"][0],
            "status": "final" if report["status"] in ("final", "corrected", "amended") else "preliminary",
            "type": {"coding": [{"system": SCT, "code": "721981007", "display": "Diagnostic studies report"},
                                {"system": LOINC, "code": "11502-2", "display": "Laboratory report"}]},
            "category": [{"coding": [{"system": SCT, "code": specialty[0], "display": specialty[1]}]}],
            "subject": {"reference": report["subject"]["reference"],
                        "identifier": {"system": NHS_NUMBER, "value": nhs_number}},
            "date": report["issued"],
            "author": [lab_ref],
            "title": "Laboratory Report",
            "attester": [{"mode": "official", "party": lab_ref}],
            "custodian": lab_ref,
            "section": sections,
        }

        # 3. link the report to the Composition; mark test groups as organizers
        report.setdefault("extension", []).append({"url": COMPOSITION_R5, "valueReference": {"reference": composition_url}})
        report["category"].append({"coding": [{"system": SCT, "code": specialty[0], "display": specialty[1]}]})
        for r in resources.values():
            if r["resourceType"] == "Observation":
                r["category"].append({"coding": [{"system": SCT, "code": specialty[0], "display": specialty[1]}]})
                if r.get("hasMember"):
                    r.setdefault("extension", []).append({"url": ORGANIZER, "valueBoolean": True})

        # 4. Provenance - which system sent the document
        agent_value = f"{PRODUCT_ID}:{lab['identifier'][0]['value']}"
        provenance = {"resourceType": "Provenance", "id": doc_id("provenance", composition_id),
                      "meta": {"profile": [NHSE_PROFILE["Provenance"]]},
                      "text": narrative(f"<p>Laboratory report {e(report['identifier'][0]['value'])} sent by product "
                                        f"{e(agent_value)}.</p>"),
                      "target": [{"reference": composition_url}],
                      "recorded": message["timestamp"],
                      "activity": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/iso-21089-lifecycle",
                                               "code": "transform"}]},
                      "agent": [{"who": {"identifier": {
                          "system": "https://api.service.nhs.uk/pathology-laboratory-reporting/test-reporting-system",
                          "value": agent_value}}}]}

        # 5. NHS England meta.profile, Composition first
        for r in resources.values():
            r["meta"] = {"profile": [NHSE_PROFILE[r["resourceType"]]]}
        entries = ([{"fullUrl": composition_url, "resource": composition}]
                   + [{"fullUrl": u, "resource": r} for u, r in resources.items()]
                   + [{"fullUrl": f"urn:uuid:{provenance['id']}", "resource": provenance}])
        bundle_id = doc_id("bundle", composition_id)
        suffix = "" if len(header["focus"]) == 1 else f"-{len(documents) + 1}"
        documents.append({"resourceType": "Bundle", "id": f"Document-{name}{suffix}",  # readable id for the IG page
                          "meta": {"profile": [NHSE_PROFILE["Bundle"]]},
                          "identifier": {"system": "urn:ietf:rfc:3986", "value": f"urn:uuid:{bundle_id}"},
                          "type": "document", "timestamp": message["timestamp"], "entry": entries})
    return documents

docs = to_document(messages["LFT-UandE-Report-Example"], "LFT-UandE-Report-Example")
[(en["resource"]["resourceType"]) for en in docs[0]["entry"]]

['Composition',
 'DiagnosticReport',
 'ServiceRequest',
 'ServiceRequest',
 'Patient',
 'Organization',
 'Specimen',
 'Observation',
 'Observation',
 'PractitionerRole',
 'Observation',
 'Observation',
 'Observation',
 'Observation',
 'Observation',
 'Observation',
 'Observation',
 'Observation',
 'Observation',
 'Practitioner',
 'Organization',
 'Provenance']

## 4. What the reader sees

The generated narrative for the LFT and U&E report - the Composition header, then one section per test group.

In [5]:
from IPython.display import HTML

def render(document):
    comp = document["entry"][0]["resource"]
    parts = [comp["text"]["div"]]
    for s in comp["section"]:
        parts.append(f"<h4>{e(s['title'])}</h4>{s['text']['div']}")
    return HTML("".join(parts))

render(docs[0])

The same narrative as XHTML, for one section:

In [6]:
print(docs[0]["entry"][0]["resource"]["section"][1]["text"]["div"][:1200])

<div xmlns="http://www.w3.org/1999/xhtml"><table><tr><th>Test</th><th>Result</th><th>Units</th><th>Reference range</th><th>Flag</th></tr><tr><td>Serum urea level</td><td>3.7</td><td>mmol/L</td><td>2.5 - 7.8</td><td></td></tr><tr><td>Serum sodium level</td><td>139</td><td>mmol/L</td><td>133 - 146</td><td></td></tr><tr><td>Serum potassium level</td><td>4.7</td><td>mmol/L</td><td>3.5 - 5.3</td><td></td></tr><tr><td>Serum creatinine level</td><td>73</td><td>umol/L</td><td>59 - 104</td><td></td></tr><tr><td>eGFR (estimated glomerular filtration rate) using creatinine Chronic Kidney Disease Epidemiology Collaboration equation per 1.73 square metres</td><td>100</td><td>mL/min/1.73m2</td><td></td><td></td></tr></table><p><i>eGFR (estimated glomerular filtration rate) using creatinine Chronic Kidney Disease Epidemiology Collaboration equation per 1.73 square metres</i>: Calculated eGFR is not valid in children or pregnancy.<br/>Less accurate with acute change in creatinine (AKI).<br/>Caution wi

## 5. Convert all the examples

One document per DiagnosticReport, written to `examples/fhir/document/<example>.json`, with the NHS England
checks from the API specification.

In [7]:
def check(document):
    """The POST /FHIR/R4/Bundle rules from the NHS England API specification."""
    entries = [en["resource"] for en in document["entry"]]
    by_url = {en["fullUrl"]: en["resource"] for en in document["entry"]}
    report = next(r for r in entries if r["resourceType"] == "DiagnosticReport")
    def requester_ods(sr):
        who = by_url[sr["requester"]["reference"]]
        org = by_url[who["organization"]["reference"]] if who["resourceType"] == "PractitionerRole" else who
        return next(i["value"] for i in org["identifier"] if i["system"] == ODS)
    ods_codes = {requester_ods(by_url[b["reference"]]) for b in report["basedOn"]}
    return {
        "type document": document["type"] == "document",
        "Composition first": entries[0]["resourceType"] == "Composition",
        "one DiagnosticReport": sum(r["resourceType"] == "DiagnosticReport" for r in entries) == 1,
        "Provenance agent": any(r["resourceType"] == "Provenance" and r["agent"][0]["who"]["identifier"]["value"] for r in entries),
        "one requesting ODS code": len(ods_codes) == 1,
        "NHS Number on Composition": entries[0]["subject"]["identifier"]["system"] == NHS_NUMBER,
        "meta.profile on all": all(r.get("meta", {}).get("profile") for r in entries),
    }

for name, message in messages.items():
    for i, document in enumerate(to_document(message, name)):
        suffix = "" if i == 0 else f"-{i + 1}"
        (OUT_DIR / f"{name}{suffix}.json").write_text(json.dumps(document, indent=2, ensure_ascii=False) + "\n")
        failed = [k for k, ok in check(document).items() if not ok]
        print(f"{name + suffix:<40} entries={len(document['entry']):>2}  {'OK' if not failed else 'FAILED ' + ', '.join(failed)}")

CRP-Report-Example                       entries=11  OK
FBC-Report-Example                       entries=24  OK
GTT-Unstructured-Report-Example          entries=13  OK
HBsAg-Report-Example                     entries=11  OK
HPV-Report-Example                       entries=11  OK
HbA1c-Report-Example                     entries=11  OK
LFT-UandE-Report-Example                 entries=22  OK
Lipids-HbA1c-Report-Example              entries=17  OK
Urine-MCS-Unstructured-Report-Example    entries=11  OK


## 6. Validate

Two checks, if the HL7 FHIR validator is available (`VALIDATOR_JAR`):

1. against the **NHS England** profiles named in `meta.profile`, using the `fhir.r4.nhsengland.pathology` package
   (the validator downloads it and its dependencies, including UK Core STU2)
2. against this guide's [Bundle - Laboratory Report Document](https://kevinmayfield.github.io/PathologyAndLaboratory/StructureDefinition-BundleLaboratoryReportDocument.html)
   (`-profile`), using only the guide's package (`output/package.tgz`), which includes the HL7 UK
   `Observation.organizer` extension definition used on test groups

In [8]:
VALIDATOR = os.getenv("VALIDATOR_JAR", str(ROOT / "validator_cli.jar"))
PACKAGE = ROOT / "output" / "package.tgz"
# Terminology server - tx.fhir.org by default (slow: several minutes per Bundle). Set VALIDATOR_TX=n/a
# to skip terminology checks; the SNOMED CT and LOINC codes are then not checked.
TX = os.getenv("VALIDATOR_TX", "http://tx.fhir.org")
print("terminology server:", TX)

def validate(paths, igs, extra=()):
    out = ROOT / ".cache" / "validation-documents.json"
    out.parent.mkdir(exist_ok=True)
    cmd = ["java", "-jar", VALIDATOR, *map(str, paths), "-version", "4.0.1",
           *[a for ig in igs for a in ("-ig", str(ig))], "-tx", TX, *([] if TX == "n/a" else ["-sct", "uk"]), "-output", str(out), *extra]
    subprocess.run(cmd, capture_output=True, text=True)
    return json.loads(out.read_text())

# England-Bundle-Lab-Pathology 0.1.0 (the Bundle profile the API specification names) adds a slice
# provenanceTestReportingSystem typed as Resource, so every entry matches it ("Element matches more than one
# slice"). The NHS England examples avoid this by declaring Bundle-eu-lab|2.0.0. These errors come from the
# published profile, not from the documents, so they are counted separately.
KNOWN_ISSUE = "provenanceTestReportingSystem"

def summarise(result, show=5):
    outcomes = result["entry"] if result.get("resourceType") == "Bundle" else [{"resource": result}]
    for en in outcomes:
        oo = en["resource"]
        source = next((x["valueString"] for x in oo.get("extension", []) if x["url"].endswith("operationoutcome-file")), "")
        errors = [i for i in oo.get("issue", []) if i["severity"] in ("error", "fatal")]
        known = [i for i in errors if KNOWN_ISSUE in i["details"]["text"]]
        errors = [i for i in errors if i not in known]
        warnings = [i for i in oo.get("issue", []) if i["severity"] == "warning"]
        print(f"{Path(source).name:<45} errors={len(errors):<3} known NHS England profile issue={len(known):<3} warnings={len(warnings)}")
        for i in errors[:show]:
            print("    ERROR", i.get("expression", [""])[0], "-", i["details"]["text"][:220])

documents = sorted(OUT_DIR.glob("*.json"))
if Path(VALIDATOR).exists():
    print("1. NHS England profiles")
    summarise(validate(documents, ["fhir.r4.nhsengland.pathology#0.3.0"]))
    if PACKAGE.exists():
        print("\n2. This guide's Bundle - Laboratory Report Document")
        # the guide's package includes the Observation.organizer extension definition (imported from HL7 UK)
        summarise(validate(documents, [PACKAGE],
                           ["-profile", "https://examplesihe.org.uk/StructureDefinition/BundleLaboratoryReportDocument"]))
else:
    print("Skipped: set VALIDATOR_JAR to validate")

terminology server: n/a
1. NHS England profiles


CRP-Report-Example.json                       errors=0   known NHS England profile issue=12  warnings=13
FBC-Report-Example.json                       errors=0   known NHS England profile issue=25  warnings=26
GTT-Unstructured-Report-Example.json          errors=0   known NHS England profile issue=14  warnings=15
HBsAg-Report-Example.json                     errors=0   known NHS England profile issue=12  warnings=13
HPV-Report-Example.json                       errors=0   known NHS England profile issue=12  warnings=13
HbA1c-Report-Example.json                     errors=0   known NHS England profile issue=12  warnings=14
LFT-UandE-Report-Example.json                 errors=0   known NHS England profile issue=23  warnings=47
Lipids-HbA1c-Report-Example.json              errors=0   known NHS England profile issue=18  warnings=23
Urine-MCS-Unstructured-Report-Example.json    errors=0   known NHS England profile issue=12  warnings=13

2. This guide's Bundle - Laboratory Report Document


CRP-Report-Example.json                       errors=0   known NHS England profile issue=0   warnings=22
FBC-Report-Example.json                       errors=0   known NHS England profile issue=0   warnings=48
GTT-Unstructured-Report-Example.json          errors=0   known NHS England profile issue=0   warnings=24
HBsAg-Report-Example.json                     errors=0   known NHS England profile issue=0   warnings=22
HPV-Report-Example.json                       errors=0   known NHS England profile issue=0   warnings=22
HbA1c-Report-Example.json                     errors=0   known NHS England profile issue=0   warnings=23
LFT-UandE-Report-Example.json                 errors=0   known NHS England profile issue=0   warnings=66
Lipids-HbA1c-Report-Example.json              errors=0   known NHS England profile issue=0   warnings=37
Urine-MCS-Unstructured-Report-Example.json    errors=0   known NHS England profile issue=0   warnings=22


## 7. Sending it

Not run here - sending needs an NHS England onboarded application and an access token (application-restricted
signed JWT authentication). The request is a FHIR JSON POST with the two headers the API requires:

```python
import requests
response = requests.post(
    "https://int.api.service.nhs.uk/pathology-laboratory-reporting/FHIR/R4/Bundle",
    headers={"Authorization": f"Bearer {access_token}",
             "Content-Type": "application/fhir+json",
             "X-Request-ID": str(uuid.uuid4()),        # unique per request; the same value when retrying
             "X-Correlation-ID": str(uuid.uuid4())},   # the same for every request in one business transaction
    data=json.dumps(document))
```

The GP system's acknowledgement is later retrieved with `GET /FHIR/R4/Bundle/{id}` - the national equivalent
of `ACK^R01`.